In [1]:
!pip install -q yfinance

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from models.wmt_model import WMTTradingModel
from models.nvda_model import NVDATradingModel
from models.mpc_model import MPCTradingModel
from models.xom_model import XOMTradingModel
from models.VAR_model import VARTradingModel

from utils import ForecastingMetrics, TradingMetrics, PortfolioEvaluator
from backtest import Backtest
import yfinance as yf
import numpy as np
import pandas as pd


from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import plotting
import matplotlib.pyplot as plt
import pandas as pd
from pypfopt.expected_returns import mean_historical_return
from pypfopt.risk_models import CovarianceShrinkage

/opt/conda/lib/python3.12/site-packages/cupy/_environment.py:541: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy, cupy-cuda12x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''


The best models for each stock are:
- WMT: VAR(0)
    - Provided lower error, higher robustness, and simpler logic
- MPC: ARIMA/ARIMA-GARCH
    - Captured both trend and shock-adjustment behavior
    - Although the MSE is slightly lower than the Naive Baseline, it has higher directional accuracy.
- NVDA: VAR(0)
    - This stock is very volatile 
    - More event-driven rather than dependent on historical data so 
        - enforcing lag-based structure increased forecast error 
    - The mean forecast (VAR(0)) minimized cumulative prediction error better than models trying to impose structure where none existed
- XOM: VAR(1)
    - Provided the best balance between low forecast error, directional accuracy, and alignment

In [4]:
stock_categories = {
    "AAPL": "Technology", "MSFT": "Technology", "NVDA": "Technology",
    "AMZN": "Technology", "GOOGL": "Technology", "META": "Technology", "TSLA": "Technology",
    "JPM": "Finance", "BAC": "Finance", "WFC": "Finance", "C": "Finance", "GS": "Finance", "MS": "Finance",
    "KO": "Consumer Goods", "PG": "Consumer Goods", "PEP": "Consumer Goods",
    "WMT": "Consumer Goods", "COST": "Consumer Goods",
    "CL": "Consumer Goods", "XOM": "Energy", "CVX": "Energy", "COP": "Energy",
    "SLB": "Energy", "EOG": "Energy", "MPC": "Energy",
    "SPY": "ETF", "QQQ": "ETF", "DIA": "ETF", "IWM": "ETF", "VTI": "ETF"
}

# Get the inverse of stock_categories
category_stock = {}
for stock, category in stock_categories.items():
    if category not in category_stock:
        category_stock[category] = []
    category_stock[category].append(stock)

In [5]:
# Define date range
start = "2022-01-01"
end = "2025-01-01"

# Download historical data
close_historical_df = yf.download(list(stock_categories.keys()), start=start, end=end, auto_adjust=True)["Close"]
close_historical_df.head()

[*********************100%***********************]  30 of 30 completed


Ticker,AAPL,AMZN,BAC,C,CL,COP,COST,CVX,DIA,EOG,...,PEP,PG,QQQ,SLB,SPY,TSLA,VTI,WFC,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2022-01-03,178.270325,170.404495,41.931023,54.393017,76.912796,63.700027,540.298584,101.482430,341.154083,75.843277,...,154.084091,147.359772,392.184113,29.220688,453.210419,399.926666,230.069687,45.828430,45.858704,55.125237
2022-01-04,176.007782,167.522003,43.574486,54.815411,77.167389,66.463226,537.934082,103.328957,343.197205,79.329285,...,154.306778,147.875412,387.097290,30.639349,453.058594,383.196655,229.634140,47.653259,45.018562,57.198719
2022-01-05,171.325989,164.356995,42.839012,54.177517,77.485611,65.323418,524.291138,104.001205,339.670746,77.873314,...,154.832367,148.544800,375.205231,30.639349,444.358856,362.706665,224.653427,47.237701,45.627270,57.910126
2022-01-06,168.465973,163.253998,43.701595,55.953270,76.976425,67.775734,524.176636,104.886169,338.084747,79.470688,...,154.867981,147.296448,374.941650,31.367104,443.941467,354.899994,224.596634,48.448235,45.500458,59.272205
2022-01-07,168.632477,162.554001,44.654987,56.703209,76.776413,69.632256,511.191284,106.392319,338.010162,81.667137,...,155.063950,147.215042,370.879944,32.269890,442.186371,342.320007,223.573944,49.478081,45.934792,59.758041


In [6]:
# def forecast_and_markowitz(as_of_date: str,
#                            horizon: int = 60,
#                            lookback_days: int = 252,
#                            objective: str = "gmir"
#                            ) -> tuple[pd.DataFrame, pd.Series]:
#     """
#     For any date after Jan 2025, re-optimize the portfolio using
#     forecasted returns from the best model per stock.

#     Parameters
#     ----------
#     as_of_date : str
#         Date string "YYYY-MM-DD". Use any date >= "2025-01-01".
#         All data strictly before this date is used for training.
#     horizon : int, default=60
#         Number of future trading days to forecast.
#     lookback_days : int, default=252
#         Number of past trading days used for model estimation.
#     objective : {"gmir", "gmv"}, default="gmir"
#         Markowitz objective: max information ratio or min variance.

#     Returns
#     -------
#     forecast_df : pd.DataFrame
#         Shape (horizon, 4) with columns ["WMT", "NVDA", "MPC", "XOM"]
#         containing forecasted daily returns.
#     weights : pd.Series
#         Markowitz optimal weights indexed by ticker, summing to 1.
#     """
#     tickers = ["WMT", "NVDA", "MPC", "XOM"]
#     model_map = {
#         "WMT": WMTTradingModel,          # VAR(0) 
#         "NVDA": NVDATradingModel,        # VAR(0)
#         "MPC": MPCTradingModel,          # ARIMA(2,1,1)/ARIMA-GARCH
#         "XOM": XOMTradingModel,          # VAR(1)
#     }

#     end = pd.to_datetime(as_of_date)
#     start = end - pd.tseries.offsets.BDay(int(lookback_days * 1.5))

#     forecasts_list: list[np.ndarray] = []

#     for t in tickers:
#         # 1) download prices up to as_of_date
#         data = yf.download(t, start=start, end=end, auto_adjust=True)
#         closes = data["Close"].dropna().tail(lookback_days)

#         # 2) compute log price
#         log_price = np.log(closes).values

#         # 3) fit best model and produce 60-day forecast
#         model_cls = model_map[t]
#         model = model_cls()
#         model.fit(log_price)

#         dummy_X = np.zeros(horizon)
#         fc = np.asarray(model.predict(dummy_X), dtype=float)
#         forecasts_list.append(fc)

#     # 4) stack forecasts of log prices into (horizon, n_assets) matrix
#     log_price_forecast_matrix = np.column_stack(forecasts_list)
#     log_price_forecast_df = pd.DataFrame(log_price_forecast_matrix, columns=tickers)

#     price_forecast_df = np.exp(log_price_forecast_df)


#     # 5) compute Markowitz weights using forecasted returns
#     mu = mean_historical_return(price_forecast_df, log_returns=True)
#     Sigma = CovarianceShrinkage(price_forecast_df, log_returns=True).ledoit_wolf()
#     ef_tan = EfficientFrontier(mu, Sigma)
#     tangency_portfolio = ef_tan.max_sharpe(risk_free_rate=0.02)
#     weights_df = pd.DataFrame(
#         [(stock, weight) for stock, weight in ef_tan.clean_weights().items()],
#         columns=['Stock', 'Weight']
#     )

#     return price_forecast_df, weights_df

In [8]:
def forecast_and_markowitz(as_of_date: str,
                           horizon: int = 60,
                           lookback_days: int = 252,
                           objective: str = "gmir"
                           ) -> tuple[pd.DataFrame, pd.DataFrame]:

    tickers = ["WMT", "NVDA", "MPC", "XOM"]
    model_map = {
        "WMT": WMTTradingModel,          # VAR(0)
        "NVDA": NVDATradingModel,        # VAR(0)
        "MPC": MPCTradingModel,          # ARIMA(0,1,0)
        "XOM": XOMTradingModel,          # VAR(1)-style AR(1)
    }

    end = pd.to_datetime(as_of_date)
    start = end - pd.tseries.offsets.BDay(int(lookback_days * 1.5))

    forecasts_list: list[np.ndarray] = []

    for t in tickers:
        # 1) download prices up to as_of_date
        data = yf.download(t, start=start, end=end, auto_adjust=True)
        closes = data["Close"].dropna().tail(lookback_days)

        # 2) compute log price
        log_price = np.log(closes).values

        # 3) fit best model and produce horizon-day forecast
        model_cls = model_map[t]
        model = model_cls()
        model.fit(log_price)

        if hasattr(model, "forecast"):
            # statistical models: ARIMA(0,1,0) for MPC, VAR(1)-style for XOM
            fc = np.asarray(model.forecast(horizon), dtype=float)
        else:
            # WMT and NVDA: use predict(len(X)) interface
            dummy_X = np.zeros(horizon)
            fc = np.asarray(model.predict(dummy_X), dtype=float)

        forecasts_list.append(fc)

    # 4) stack forecasts of log prices into (horizon, n_assets) matrix
    log_price_forecast_matrix = np.column_stack(forecasts_list)
    log_price_forecast_df = pd.DataFrame(log_price_forecast_matrix, columns=tickers)

    # convert back to price level forecasts
    price_forecast_df = np.exp(log_price_forecast_df)

    # 5) compute Markowitz weights using forecasted returns
    mu = mean_historical_return(price_forecast_df, log_returns=True)
    Sigma = CovarianceShrinkage(price_forecast_df, log_returns=True).ledoit_wolf()
    ef_tan = EfficientFrontier(mu, Sigma)
    tangency_portfolio = ef_tan.max_sharpe(risk_free_rate=0.02)
    weights_df = pd.DataFrame(
        [(stock, weight) for stock, weight in ef_tan.clean_weights().items()],
        columns=['Stock', 'Weight']
    )

    return price_forecast_df, weights_df

In [7]:
# #forecast_df, weights = forecast_and_markowitz("2025-02-01")
# forecast_df, weights = forecast_and_markowitz("2025-02-01")

# print("First 5 of the 60-day forecasts:")
# display(forecast_df.head())

# print("\nMarkowitz weights from forecasted returns:")
# print(weights)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


AttributeError: 'MPCTradingModel' object has no attribute 'predict'

In [9]:
forecast_df, weights = forecast_and_markowitz("2025-02-01")
display(forecast_df.head())
print(weights)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


,WMT,NVDA,MPC,XOM
0,71.508766,111.544783,142.544815,103.275343
1,71.508766,111.544783,142.544815,103.494792
2,71.508766,111.544783,142.544815,103.706607
3,71.508766,111.544783,142.544815,103.911039
4,71.508766,111.544783,142.544815,104.108330


  Stock  Weight
0   WMT     0.0
1  NVDA     0.0
2   MPC     0.0
3   XOM     1.0


In [10]:
mu = mean_historical_return(forecast_df, log_returns=True)
Sigma = CovarianceShrinkage(forecast_df, log_returns=True).ledoit_wolf()

# Compute tangency portfolio
ef_tan = EfficientFrontier(mu, Sigma)
tangency_portfolio = ef_tan.max_sharpe(risk_free_rate=0.02)

weights_df = pd.DataFrame(
    [(stock, weight) for stock, weight in ef_tan.clean_weights().items()],
    columns=['Stock', 'Weight']
)

weights_df

,Stock,Weight
0,WMT,0.0
1,NVDA,0.0
2,MPC,0.0
3,XOM,1.0
